# Análise exploratória — Imóveis em São Paulo

Objetivo desta primeira etapa: **conhecer a base antes de modelar**. Vamos responder, em ordem:

1. Qual o tamanho e o formato da base?
2. Os tipos das colunas fazem sentido?
3. Há valores nulos?
4. Há linhas duplicadas?
5. A coluna `Valor` mistura coisas diferentes?
6. Há outliers impossíveis?

Cada achado vira uma decisão de limpeza, registrada no final.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("../data/dados_imoveis.csv")
df.head(10)

,Rua,Bairro,Cidade,Metragem,Quartos,Banheiros,Vagas,Valor
0,"Avenida Itacira, 255",Planalto Paulista,São Paulo,1000,4,8,6,R$ 7.000.000
1,"Rua Aurelia Perez Alvarez, 42",Jardim dos Estados,São Paulo,496,4,4,4,R$ 3.700.000
2,Rua Alba Valdez,Jardim Reimberg,São Paulo,125,4,3,2,R$ 380.000
3,NaN,Jardim Morumbi,São Paulo,310,3,2,4,R$ 685.000
4,"Rua Tobias Barreto, 195",Mooca,São Paulo,100,3,2,2,R$ 540.000
5,Rua Graham Bell,Santo Amaro,São Paulo,440,4,4,6,R$ 1.980.000
6,"Rua Francisco Paulo, 31",Cidade Mãe do Céu,São Paulo,145,4,4,2,R$ 850.000
7,"Rua Guilherme Valente, 150",Jardim Vergueiro,São Paulo,150,2,2,2,R$ 450.000
8,"Rua Sagrada Família, 8",Jardim Benfica,São Paulo,48,2,1,1,R$ 199.900
9,Rua Tapaji,Vila Alpina,São Paulo,114,3,3,2,R$ 585.000


In [ ]:
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")

Linhas: 10,008 | Colunas: 8


In [ ]:
# A base é apresentada como "Grande São Paulo", mas podemos identificar apenas um município (São Paulo) ao invés de uma região metropolitana

print(df["Cidade"].value_counts())
print("\nBairros distintos:", df["Bairro"].nunique())

Cidade
São Paulo    10008
Name: count, dtype: int64

Bairros distintos: 701


**Achado:** a base é só de São Paulo, então `Cidade` não traz informação nenhuma — pode ser descartada. São centenas de bairros.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10008 entries, 0 to 10007
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Rua        6574 non-null   str  
 1   Bairro     9996 non-null   str  
 2   Cidade     10008 non-null  str  
 3   Metragem   10008 non-null  int64
 4   Quartos    10008 non-null  int64
 5   Banheiros  10008 non-null  int64
 6   Vagas      10008 non-null  int64
 7   Valor      10008 non-null  str  
dtypes: int64(4), str(4)
memory usage: 625.6 KB


**Achado:** `Valor` aparece como texto (`object`/`str`), não como número. Antes de qualquer conta precisamos entender por quê.

In [12]:
# Amostra aleatória dos valores para ver o formato
df["Valor"].sample(15, random_state=1).tolist()

['R$ 598.000',
 'R$ 1.350.000',
 'R$ 1.200.000',
 'R$ 11.000\n                      /Mês',
 'R$ 1.648.000',
 'R$ 750.000',
 'R$ 8.000\n                      /Mês',
 'R$ 12.000.000',
 'R$ 760.000',
 'R$ 3.000.000',
 'R$ 10.000\n                      /Mês',
 'R$ 450.000',
 'R$ 1.080.000',
 'R$ 17.000\n                      /Mês',
 'R$ 650.000']

**Achado importante:** além do formato `R$ 1.350.000`, aparecem valores como `R$ 11.000 /Mês`. Ou seja, **a base mistura imóveis à venda com imóveis para aluguel**. Se não separarmos, um aluguel de R$ 8 mil seria lido como uma casa barata de R$ 8 mil.

In [14]:
# Quais sufixos de periodicidade existem? (tudo que vem depois da '/')

sufixo = df["Valor"].str.extract(r"/\s*(\w+)")[0]
print(sufixo.value_counts(dropna=False).rename({np.nan: "(sem sufixo = venda)"}))

0
(sem sufixo = venda)    9156
Mês                      848
Ano                        2
Dia                        2
Name: count, dtype: int64


In [15]:
eh_aluguel = df["Valor"].str.contains("/", na=False)
print(eh_aluguel.value_counts().rename({True: "Aluguel", False: "Venda"}))
print(f"\nAluguel = {eh_aluguel.mean():.1%} da base")

Valor
Venda      9156
Aluguel     852
Name: count, dtype: int64

Aluguel = 8.5% da base


In [16]:
# Algum valor tem formato diferente desses dois? (ex.: 'Sob consulta')
fora_do_padrao = ~df["Valor"].str.match(r"^R\$\s?[\d\.]+", na=False)
df.loc[fora_do_padrao, "Valor"].value_counts()

Series([], Name: count, dtype: int64)

In [17]:
# Converter texto -> número e marcar o tipo de negócio
def converter_valor(texto):
    if pd.isna(texto):
        return np.nan
    numero = (texto.split("/")[0]          # tira '/Mês', '/Ano', '/Dia'
                   .replace("R$", "")
                   .replace(".", "")       # separador de milhar
                   .strip())
    return pd.to_numeric(numero, errors="coerce")

df["Valor_num"] = df["Valor"].apply(converter_valor)
df["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")
df["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)

df.groupby("Tipo")["Valor_num"].describe()

,count,mean,std,min,25%,50%,75%,max
Tipo,,,,,,,,
Aluguel,852.00,"21,219.15","17,007.45","1,650.00","10,000.00","16,000.00","26,000.00","120,000.00"
Venda,"9,156.00","3,052,202.96","3,944,208.19","13,000.00","890,000.00","1,800,000.00","3,500,000.00","65,000,000.00"


In [18]:
# Converter texto -> número e marcar o tipo de negócio
def converter_valor(texto):
    if pd.isna(texto):
        return np.nan
    numero = (texto.split("/")[0]          # tira '/Mês', '/Ano', '/Dia'
                   .replace("R$", "")
                   .replace(".", "")       # separador de milhar
                   .strip())
    return pd.to_numeric(numero, errors="coerce")

df["Valor_num"] = df["Valor"].apply(converter_valor)
df["Tipo"] = np.where(eh_aluguel, "Aluguel", "Venda")
df["Periodo"] = sufixo.fillna("")  # Mês, Ano, Dia ou vazio (venda)

df.groupby("Tipo")["Valor_num"].describe()

,count,mean,std,min,25%,50%,75%,max
Tipo,,,,,,,,
Aluguel,852.00,"21,219.15","17,007.45","1,650.00","10,000.00","16,000.00","26,000.00","120,000.00"
Venda,"9,156.00","3,052,202.96","3,944,208.19","13,000.00","890,000.00","1,800,000.00","3,500,000.00","65,000,000.00"


Existe uma diferença de escala entre os 2 grupos: são populações diferentes e devem ser analisadas separadamente.